In [1]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

v = embed.encode(q)

In [2]:
v[0]

np.float64(-0.02058200593003704)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [25]:
texts = []

for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        texts.append(doc["content"])

texts

['# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is e

In [26]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
v_query = embed.encode(q)
scores = X.dot(v_query)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(0), np.float64(0.3610702814461231))

In [34]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunk_texts = [chunk["content"] for chunk in chunks]

In [41]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X_chunks = []

for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X_chunks.extend(batch_vectors)

X_chunks = np.array(X_chunks)

  0%|          | 0/6 [00:00<?, ?it/s]

In [42]:
scores = X_chunks.dot(v)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489015778372396))

In [43]:
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [44]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X_chunks, chunks)

In [46]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [48]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [51]:
from minsearch import Index, VectorSearch

tindex = Index(text_fields=["content"])
tindex.fit(chunks)

vindex = VectorSearch()
vindex.fit(X_chunks, chunks)

In [52]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

In [58]:
tresults = tindex.search(query, num_results=5)
text_files = {file["filename"] for file in tresults}

In [59]:
vresults = vindex.search(query_vector, num_results=5)
vector_files = {file["filename"] for file in vresults}

In [60]:
vector_files - text_files

{'02-vector-search/lessons/08-pgvector.md'}

In [61]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [64]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

In [75]:
text_results = tindex.search(query, num_results=5)
text_files = {file["filename"] for file in text_results}

In [76]:
vector_results = vindex.search(query_vector, num_results=5)
vector_files = {file["filename"] for file in vector_results}

In [77]:
results = rrf([vector_results, text_results])

In [78]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'